# Literature Search Harness

Structured, logged, reproducible paper search. Runs in Colab with no API key and no
sign-up. Produces three things your paper needs:

1. a **search log** (query, date, hits) as a LaTeX table
2. a **screening block** to paste into Claude for inclusion decisions
3. a **`refs.bib`** of verified entries with real DOIs

**Run order:** Cell 1 → write your criteria in Cell 2 → Cell 3 → Cell 4 → screen in Claude
→ paste decisions into Cell 6 → Cells 7 and 8.

The point is not that searching is hard. The point is that a documented search procedure is
what a research-methodology examiner is looking for, and doing it this way costs you
fifteen minutes.

## 1. Setup

In [1]:
import requests, re, json, time, textwrap, hashlib
from datetime import date, datetime
import pandas as pd

# OpenAlex asks for an email in requests -- it buys you the faster "polite pool".
EMAIL = 'rkiri98@gmail.com'

# Mount Drive FIRST, so exports land beside paper.tex and survive a runtime reset.
# Without this, the path below is just a folder inside the container and everything
# written to it disappears silently when Colab disconnects.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive not mounted:", e, "\n-> exports will NOT persist; download them manually")

PROJECT = '/content/drive/MyDrive/rm_paper'
os.makedirs(PROJECT, exist_ok=True)
if not os.path.ismount('/content/drive'):
    print("WARNING: /content/drive is not mounted. Files written to PROJECT are temporary.")

SEARCH_LOG = []   # one row per query executed
print("ready", date.today().isoformat())

Mounted at /content/drive
ready 2026-09-21


## 2. Inclusion criteria — write these BEFORE searching

This cell is the whole methodological point. Criteria written *after* seeing results are
post-hoc rationalization, and an examiner who asks "where did these criteria come from?"
will find out in about one question.

Four lines is enough. Be specific enough that someone else applying them to the same hits
would include the same papers.

In [2]:
CRITERIA = {
    'population':  'hospitalized adult patients, structured Electronic Health Records (EHR)',
    'construct':   'predicting hospital readmission or 30-day readmission risk',
    'method':      'machine learning, statistical modeling, or feature engineering on tabular clinical data',
    'recency':     'published 2015 or later',
}

EXCLUDE = [
    'non-clinical datasets or purely qualitative/narrative frameworks without empirical ML models',
    'medical imaging or genomic-only studies lacking structured EHR/tabular records',
    'studies without performance metrics (e.g., ROC-AUC, F1-score, Recall)',
]

for k, v in CRITERIA.items():
    print(f"  INCLUDE  {k:12s} {v}")
for e in EXCLUDE:
    print(f"  EXCLUDE  {e}")

  INCLUDE  population   hospitalized adult patients, structured Electronic Health Records (EHR)
  INCLUDE  construct    predicting hospital readmission or 30-day readmission risk
  INCLUDE  method       machine learning, statistical modeling, or feature engineering on tabular clinical data
  INCLUDE  recency      published 2015 or later
  EXCLUDE  non-clinical datasets or purely qualitative/narrative frameworks without empirical ML models
  EXCLUDE  medical imaging or genomic-only studies lacking structured EHR/tabular records
  EXCLUDE  studies without performance metrics (e.g., ROC-AUC, F1-score, Recall)


## 3. Search

`openalex_search` hits the OpenAlex API — free, keyless, ~250M works. Every call appends a
row to `SEARCH_LOG`, which becomes your search-log table. Vary the query wording: three or
four phrasings catch what any single one misses, and the variation is itself evidence of a
systematic search.

In [3]:
def _abstract(work):
    '''OpenAlex ships abstracts as an inverted index; rebuild the text.'''
    inv = work.get('abstract_inverted_index')
    if not inv:
        return ''
    positions = []
    for word, idxs in inv.items():
        for i in idxs:
            positions.append((i, word))
    positions.sort()
    return ' '.join(w for _, w in positions)


def _citekey(work):
    authorships = work.get('authorships') or []
    if authorships:
        name = authorships[0].get('author', {}).get('display_name', 'anon')
        last = re.sub(r'\W+', '', name.split()[-1].lower())
    else:
        last = 'anon'
    return f"{last}{work.get('publication_year', 'nd')}"


def openalex_search(query, n=25, from_year=2005, log=True):
    params = {
        'search': query,
        'per-page': min(n, 200),
        'mailto': EMAIL,
        'filter': f'from_publication_date:{from_year}-01-01',
        'sort': 'relevance_score:desc',
    }
    r = requests.get('https://api.openalex.org/works', params=params, timeout=40)
    r.raise_for_status()
    payload = r.json()
    rows = []
    for rank, w in enumerate(payload.get('results', [])):
        authorships = w.get('authorships') or []
        venue = ((w.get('primary_location') or {}).get('source') or {}).get('display_name', '')
        rows.append({
            'rank':     rank,                        # relevance position in THIS query
            'relevance': w.get('relevance_score', 0.0) or 0.0,
            'key':      _citekey(w),
            'title':    (w.get('title') or '').strip(),
            'year':     w.get('publication_year'),
            'authors':  ' and '.join(a.get('author', {}).get('display_name', '')
                                     for a in authorships[:8]),
            'venue':    venue or '',
            'doi':      (w.get('doi') or '').replace('https://doi.org/', ''),
            'cited_by': w.get('cited_by_count', 0),
            'oa_url':   ((w.get('best_oa_location') or {}) or {}).get('pdf_url') or '',
            'abstract': _abstract(w),
            'query':    query,
        })
    if log:
        SEARCH_LOG.append({
            'query':     query,
            'source':    'OpenAlex',
            'date':      date.today().isoformat(),
            'time':      datetime.now().strftime('%H:%M'),
            'filter':    f'>= {from_year}',
            'returned':  payload.get('meta', {}).get('count', len(rows)),
            'retrieved': len(rows),
        })
    return rows

In [4]:
QUERIES = [
    'hospital readmission prediction machine learning EHR',
    'predicting 30 day readmission missing data class imbalance',
    'clinical dataset machine learning patient demographics readmission',
    'electronic health records readmission feature engineering machine learning',
]

hits = []
for q in QUERIES:
    try:
        found = openalex_search(q, n=25)
        hits += found
        print(f"{len(found):3d} retrieved   {q}")
    except Exception as e:
        print(f"FAILED  {q}  ->  {type(e).__name__}: {e}")
    time.sleep(1)

raw = pd.DataFrame(hits)
print(f"\n{len(raw)} rows before dedupe")

 25 retrieved   hospital readmission prediction machine learning EHR
 25 retrieved   predicting 30 day readmission missing data class imbalance
 25 retrieved   clinical dataset machine learning patient demographics readmission
 25 retrieved   electronic health records readmission feature engineering machine learning

100 rows before dedupe


## 4. Dedupe and rank

Deduplicate on DOI where present, falling back to a normalized title. The before/after
counts go into your search log — that is the PRISMA-style number an examiner recognizes.

**Ordering matters more than it looks.** Results come back ranked by relevance to your
query. Re-sorting by citation count destroys that and floats famous-but-unrelated papers
to the top — a highly cited epidemiology cohort study will outrank exactly the small
experiment you need. So rank by best relevance position, and keep citations as a
tiebreaker only.

In [5]:
def _norm_title(t):
    return re.sub(r'\W+', '', (t or '').lower())

def dedupe(df):
    df = df.copy()
    df['_id'] = df.apply(
        lambda r: r['doi'].lower() if r['doi'] else _norm_title(r['title']), axis=1)
    before = len(df)
    # a paper found by several queries keeps its BEST rank and counts as corroborated
    agg = df.groupby('_id').agg(best_rank=('rank', 'min'),
                                n_queries=('query', 'nunique'))
    df = df.drop_duplicates('_id', keep='first').merge(agg, on='_id').drop(columns='_id')
    print(f"{before} -> {len(df)} after dedupe ({before - len(df)} duplicates removed)")
    # relevance first; papers hit by multiple queries rank higher; citations break ties
    df = df.sort_values(['best_rank', 'n_queries', 'cited_by'],
                        ascending=[True, False, False])
    return df.reset_index(drop=True)

papers = dedupe(raw)
COUNTS = {'retrieved': len(raw), 'after_dedupe': len(papers)}
papers[['key', 'year', 'best_rank', 'n_queries', 'cited_by', 'title']].head(20)

100 -> 77 after dedupe (23 duplicates removed)


,key,year,best_rank,n_queries,cited_by,title
0,miotto2017,2017,0,2,3141,"Deep learning for healthcare: review, opportun..."
1,rajkomar2018,2018,0,2,2614,Scalable and accurate deep learning with elect...
2,johnson2023,2023,0,1,3289,"MIMIC-IV, a freely accessible electronic healt..."
3,awan2019,2019,0,1,153,Machine Learning-Based Prediction of Heart Fai...
4,huang2021,2021,1,3,138,Application of machine learning in predicting ...
5,kavakiotis2017,2017,1,2,1469,Machine Learning and Data Mining Methods in Di...
6,huang2019,2019,1,1,625,ClinicalBERT: Modeling Clinical Notes and Pred...
7,hripcsak2012,2012,2,1,846,Next-generation phenotyping of electronic heal...
8,mahmoudi2020,2020,2,1,194,Use of electronic medical records in developme...
9,xiao2018,2018,3,2,848,Opportunities and challenges in developing dee...


### Optional keyword pre-filter

Keyword search is noisy. If your candidate list is full of obviously unrelated work, this
cuts it before you spend a Claude message on screening.

Each inner list is an OR group; a paper must match **every** group to survive. Two groups
— your outcome and your manipulation — is usually enough. Log the counts: a pre-filter is
a screening step and belongs in the record.

In [6]:
MUST_MATCH = [
    ['dishonest', 'cheat', 'lying', 'lie ', 'deception', 'honesty', 'fraud'],
    ['anonym', 'observ', 'watch', 'monitor', 'surveill', 'unseen', 'privacy'],
]

def prefilter(df, groups):
    hay = (df['title'].fillna('') + ' ' + df['abstract'].fillna('')).str.lower()
    keep = pd.Series(True, index=df.index)
    for g in groups:
        keep &= hay.apply(lambda t: any(term in t for term in g))
    out = df[keep].reset_index(drop=True)
    print(f"{len(df)} -> {len(out)} after keyword pre-filter "
          f"({len(df) - len(out)} dropped as topically unrelated)")
    return out

# papers = prefilter(papers, MUST_MATCH)
# COUNTS['after_prefilter'] = len(papers)
# papers[['key', 'year', 'best_rank', 'cited_by', 'title']].head(20)

## 5. Screening block

Prints every candidate as a numbered record. Copy the whole output into Claude in one
message, together with your criteria from Cell 2, and ask for an include/exclude decision
plus a one-line reason for each.

One message, not forty. Batch screening is the single biggest saving on your Pro usage.

In [7]:
def screening_block(df, max_abstract=900):
    out = []
    for i, r in df.iterrows():
        abst = (r['abstract'] or '')[:max_abstract] or '(no abstract available)'
        out.append(
            f"[{i}] {r['title']} ({r['year']}, cited {r['cited_by']})\n"
            f"    venue: {r['venue']}\n"
            f"    doi:   {r['doi']}\n"
            f"    {abst}\n"
        )
    return '\n'.join(out)

block = screening_block(papers)
print(f"--- {len(papers)} candidates, {len(block)} characters ---\n")

# Screen in batches so one message stays manageable and Claude stays accurate.
BATCH = 40
for start in range(0, len(papers), BATCH):
    chunk = screening_block(papers.iloc[start:start + BATCH])
    print(f"\n{'=' * 70}\nBATCH {start}-{min(start + BATCH, len(papers)) - 1}"
          f"  ({len(chunk)} chars) -- copy this block into one Claude message\n{'=' * 70}\n")
    print(chunk)

--- 77 candidates, 75118 characters ---


BATCH 0-39  (39572 chars) -- copy this block into one Claude message

[0] Deep learning for healthcare: review, opportunities and challenges (2017, cited 3141)
    venue: Briefings in Bioinformatics
    doi:   10.1093/bib/bbx044
    Gaining knowledge and actionable insights from complex, high-dimensional and heterogeneous biomedical data remains a key challenge in transforming health care. Various types of data have been emerging in modern biomedical research, including electronic health records, imaging, -omics, sensor data and text, which are complex, heterogeneous, poorly annotated and generally unstructured. Traditional data mining and statistical learning approaches typically need to first perform feature engineering to obtain effective and more robust features from those data, and then build prediction or clustering models on top of them. There are lots of challenges on both steps in a scenario of complicated data and lacking of sufficien

### The screening prompt

Paste the block above into Claude together with this. The pipe-delimited output format
matters — Cell 6 parses it, so you never retype a decision.

```
Here are my pre-registered inclusion criteria:
<paste CRITERIA and EXCLUDE from Cell 2>

Below are N candidate papers retrieved from OpenAlex. Screen each one against those
criteria only. Do not use outside knowledge about the papers.

Output one line per paper, no preamble, in exactly this format:
index | include OR exclude | one-line reason citing which criterion decided it

<paste the screening block>
```

Ask for the reason on every line, including the includes. "Meets all four criteria,
lab experiment with anonymity manipulation" is a defensible sentence; a bare index is not.

## 6. Screening decisions

Paste Claude's output between the triple quotes. The parser is forgiving about spacing and
about `|`, tab or comma separators, but it needs three fields per line.

This cell is what turns a pile of papers into a documented screening procedure: it keeps
every rejection *and its reason*, which is the part an examiner can actually interrogate.

In [8]:
DECISIONS_RAW = """
0 | exclude | meta-analysis; retained for snowballing only
1 | exclude | methodological review, no dishonesty outcome measured
2 | exclude | review of tax experiments, not primary evidence
3 | exclude | manipulates beliefs about others' lying, not own observability
4 | exclude | methodological essay, no empirical data
5 | exclude | neuromarketing review, off-topic
6 | include | deception game manipulating scrutiny and identity revelation
7 | exclude | outcome is prosocial choice, not dishonesty
8 | exclude | theoretical paper on CSR, no empirical data
9 | exclude | marketing/privacy survey, off-topic
10 | exclude | no observability manipulation; predictive validity study
11 | exclude | manipulates social information, not observability
12 | exclude | theory and guidelines paper, no data
13 | exclude | self-reported faculty vigilance, no behavioural cheating measure
14 | include | watching-eyes signage field intervention, theft as dishonest behaviour
15 | exclude | microbial ecology, off-topic
16 | exclude | manipulates honesty priming, not observability
17 | exclude | perspective article, no empirical data
18 | exclude | lying experiment but observability not manipulated
19 | exclude | observability manipulated but outcome is prosociality, not dishonesty
20 | exclude | manipulates stakes, not observability
21 | exclude | privacy nudge, no dishonesty outcome
22 | exclude | review of punishment literature
23 | include | eye cues manipulated, prosocial lying measured directly
24 | exclude | philosophical paper on nudge ethics
25 | include | probability of detection manipulated, cheating task, reports distribution
26 | exclude | eye cues but outcome is public-good investment, not dishonesty
27 | include | tax authority supervision manipulated in field experiment
28 | exclude | manipulates corruption beliefs, not observability
29 | exclude | password security nudges, off-topic
30 | exclude | social loafing, no dishonesty measure
31 | exclude | food choice methodology review
32 | exclude | prosocial behaviour in prisoners, no observability manipulation
33 | exclude | manipulates commitment requests, not observability
34 | exclude | microbial ecology, off-topic
35 | exclude | blood donation incentives, off-topic
36 | exclude | review of moral psychology of AI
37 | exclude | conceptual paper on algorithmic nudging
38 | exclude | trust experiment, no dishonesty or observability manipulation
39 | exclude | eye images but outcome is helping behaviour, not dishonesty
40 | exclude | password nudging, off-topic
41 | include | die-roll task with observability explicitly manipulated
42 | exclude | manipulates pledges, not observability
43 | exclude | art perception model, off-topic
44 | exclude | review of digital choice architecture
45 | exclude | cross-country honesty data, observability not manipulated
46 | exclude | preprint duplicate of 7, prosocial outcome
47 | exclude | linguistics chapter, off-topic
48 | exclude | observational market data, no experimental manipulation
49 | include | darkness manipulates perceived anonymity, coin-toss dishonesty, pre-registered
50 | exclude | cellular biology, off-topic
51 | exclude | willful ignorance and self-image, observability not manipulated
52 | exclude | review of behaviour change interventions
53 | exclude | auditor skepticism, participants not the dishonest actors
54 | exclude | handbook survey chapter
55 | exclude | personality and stockpiling, no observability manipulation
56 | exclude | observation manipulated but outcome is generosity, not dishonesty
57 | exclude | privacy tool adoption, off-topic
58 | exclude | review article, no data
59 | exclude | honesty field experiment but manipulates moral reminder, not observability
60 | exclude | e-commerce nudge perceptions, off-topic
61 | exclude | theoretical paper on moral learning
62 | exclude | gender and risk-taking, no dishonesty measure
63 | include | webcam proctoring vs unproctored, randomized field experiment on exam cheating
64 | exclude | mobile payment security nudges, off-topic
65 | exclude | anthropology theory, off-topic
66 | exclude | preprint duplicate of 55
67 | exclude | investment nudges, off-topic
68 | exclude | participants aged 13-14, fails adult population criterion
69 | exclude | birth cohort epidemiology, off-topic
70 | include | detection probability varied in choice experiment on contract cheating
71 | exclude | voter registration, off-topic
72 | exclude | conceptual paper on nudge ethics
73 | exclude | theoretical review of moral preferences
74 | exclude | eye images but outcome is blood donation, not dishonesty
75 | exclude | duplicate survey chapter of 54
76 | exclude | survey data quality, observability not manipulated
77 | exclude | paperless billing field experiment, off-topic
78 | exclude | policy review
79 | exclude | ethics essay on surveillance, no data
80 | exclude | recycling nudges, off-topic
81 | exclude | methodological review of lab-in-field designs
82 | exclude | organizational theory, off-topic
83 | exclude | reanalysis across studies, personality not observability
84 | exclude | gender and competition, no dishonesty measure
85 | exclude | systematic review; snowballing only
86 | exclude | evolutionary theory paper
87 | include | response anonymity manipulated, validated against actual dice-game cheating
88 | exclude | manipulates scarcity, not observability
89 | exclude | local food purchasing, off-topic
90 | exclude | review; useful for snowballing on scrutiny effects
91 | exclude | LLM risk taxonomy, off-topic
92 | exclude | manipulates outcome probabilities, not observability
93 | exclude | manipulates oaths and commitment, not observability
"""

def parse_decisions(raw):
    rows, bad = [], []
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = re.split(r'\s*[|\t]\s*', line, maxsplit=2)
        if len(parts) < 3:
            parts = re.split(r'\s*,\s*', line, maxsplit=2)
        if len(parts) < 3 or not parts[0].strip().strip('[]').isdigit():
            bad.append(line); continue
        idx = int(parts[0].strip().strip('[]'))
        decision = parts[1].strip().lower()
        decision = 'include' if decision.startswith('inc') else 'exclude'
        rows.append({'index': idx, 'decision': decision, 'reason': parts[2].strip()})
    if bad:
        print(f"{len(bad)} line(s) could not be parsed -- check these:")
        for b in bad[:5]:
            print("   ", b)
    return pd.DataFrame(rows)

decisions = parse_decisions(DECISIONS_RAW)

missing = set(papers.index) - set(decisions['index'])
extra   = set(decisions['index']) - set(papers.index)
if missing:
    print(f"WARNING: {len(missing)} candidates have no decision: {sorted(missing)[:10]}")
if extra:
    print(f"WARNING: decisions reference unknown indices: {sorted(extra)[:10]}")

SCREENED = papers.join(decisions.set_index('index')[['decision', 'reason']])
included = SCREENED[SCREENED['decision'] == 'include'].reset_index(drop=True)
excluded = SCREENED[SCREENED['decision'] == 'exclude'].reset_index(drop=True)

COUNTS['screened'] = int(SCREENED['decision'].notna().sum())
COUNTS['included'] = len(included)
COUNTS['excluded'] = len(excluded)
print(f"\nscreened {COUNTS['screened']}  ->  included {COUNTS['included']}, "
      f"excluded {COUNTS['excluded']}")
included[['key', 'year', 'title', 'reason']]


screened 77  ->  included 9, excluded 68


,key,year,title,reason
0,huang2019,2019,ClinicalBERT: Modeling Clinical Notes and Pred...,deception game manipulating scrutiny and ident...
1,xu2020,2020,Federated Learning for Healthcare Informatics,"watching-eyes signage field intervention, thef..."
2,amritphale2021,2021,Predictors of 30-Day Unplanned Readmission Aft...,"eye cues manipulated, prosocial lying measured..."
3,li2019,2019,Patient clustering improves efficiency of fede...,"probability of detection manipulated, cheating..."
4,darabi2021,2021,Machine Learning-Enabled 30-Day Readmission Mo...,tax authority supervision manipulated in field...
5,shamout2020,2020,Machine Learning for Clinical Outcome Prediction,die-roll task with observability explicitly ma...
6,pham2017,2017,Predicting healthcare trajectories from medica...,"darkness manipulates perceived anonymity, coin..."
7,wang2022,2022,Nationwide hospital admission data statistics ...,"webcam proctoring vs unproctored, randomized f..."
8,jothi2015,2015,Data Mining in Healthcare – A Review,detection probability varied in choice experim...


## 7. Export `refs.bib`

Every entry here came from a real API record with a real DOI. Nothing was generated from
an LLM's memory — which is the only way to be certain you have not cited a paper that does
not exist.

In [9]:
def to_bibtex(df):
    entries = []
    for _, r in df.iterrows():
        fields = [
            ('title',   r['title']),
            ('author',  r['authors']),
            ('journal', r['venue']),
            ('year',    r['year']),
            ('doi',     r['doi']),
        ]
        body = ',\n'.join(f"  {k:7s} = {{{v}}}" for k, v in fields if v)
        entries.append('@article{' + r['key'] + ',\n' + body + '\n}')
    return '\n\n'.join(entries)

bib = to_bibtex(included)
with open(f'{PROJECT}/refs.bib', 'w') as f:
    f.write(bib)
print(f"wrote {PROJECT}/refs.bib  ({len(included)} entries)\n")
print(bib[:1200])

wrote /content/drive/MyDrive/rm_paper/refs.bib  (9 entries)

@article{huang2019,
  title   = {ClinicalBERT: Modeling Clinical Notes and Predicting Hospital Readmission},
  author  = {Kexin Huang and Jaan Altosaar and Rajesh Ranganath},
  journal = {arXiv (Cornell University)},
  year    = {2019},
  doi     = {10.48550/arxiv.1904.05342}
}

@article{xu2020,
  title   = {Federated Learning for Healthcare Informatics},
  author  = {Jie Xu and Benjamin S. Glicksberg and Chang Su and Peter Walker and Jiang Bian and Fei Wang},
  journal = {Journal of Healthcare Informatics Research},
  year    = {2020},
  doi     = {10.1007/s41666-020-00082-4}
}

@article{amritphale2021,
  title   = {Predictors of 30-Day Unplanned Readmission After Carotid Artery Stenting Using Artificial Intelligence},
  author  = {Amod Amritphale and Ranojoy Chatterjee and Suvo Chatterjee and Nupur Amritphale and Ali Rahnavard and G. Mustafa Awan and Bassam Omar and Gregg C. Fonarow},
  journal = {Advances in Therapy},
  ye

## 8. Search log and extraction table as LaTeX

`search_log.tex` documents the procedure. `extraction.tex` is your Related Work table with
the rows pre-filled — you write in the Method and Limitation columns as you read.

`\input{}` both from `paper.tex` and they stay in sync when you re-run the search.

In [10]:
def esc(s):
    s = str(s)
    for a, b in [('&', r'\&'), ('%', r'\%'), ('_', r'\_'), ('#', r'\#')]:
        s = s.replace(a, b)
    return s

def search_log_tex(log, counts):
    rows = '\n'.join(
        f"{esc(r['source'])} & {esc(r['query'])} & {esc(r['date'])} {esc(r.get('time', ''))} "
        f"& {r['returned']} & {r['retrieved']} \\\\"
        for r in log)
    return (
        "\\begin{table}[htbp]\n\\centering\n"
        "\\caption{Search log. All queries executed against the OpenAlex API.}\n"
        "\\label{tab:searchlog}\n\\small\n"
        "\\begin{tabular}{p{1.8cm}p{5.2cm}p{2.1cm}rr}\n\\toprule\n"
        "Source & Query & Executed & Hits & Retrieved \\\\\n\\midrule\n"
        + rows + "\n\\bottomrule\n\\end{tabular}\n\n"
        "\\vspace{0.5em}\n{\\small Retrieved $n="
        + str(counts.get('retrieved', 0)) + "$; after deduplication $n="
        + str(counts.get('after_dedupe', 0)) + "$; screened $n="
        + str(counts.get('screened', 0)) + "$; included $n="
        + str(counts.get('included', 0)) + "$.}\n\\end{table}\n"
    )


def screening_log_tex(screened):
    '''Full screening record -- every candidate, its decision and the reason.'''
    rows = []
    for i, r in screened.iterrows():
        if pd.isna(r.get('decision')):
            continue
        short = (r['title'][:58] + '...') if len(r['title']) > 58 else r['title']
        rows.append(f"{i} & {esc(short)} & {esc(r['decision'])} & {esc(r['reason'])} \\\\")
    return (
        "\\begin{table}[htbp]\n\\centering\n"
        "\\caption{Screening record. Every retrieved candidate, its decision and the "
        "criterion that decided it.}\n"
        "\\label{tab:screening}\n\\scriptsize\n"
        "\\begin{tabular}{rp{5.2cm}p{1.3cm}p{5.2cm}}\n\\toprule\n"
        "\\# & Title & Decision & Reason \\\\\n\\midrule\n"
        + '\n'.join(rows) + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    )

def extraction_tex(df):
    rows = '\n'.join(
        f"\\citet{{{r['key']}}} & & & \\\\" for _, r in df.iterrows())
    return (
        "\\begin{table}[htbp]\n\\centering\n"
        "\\caption{Included studies: datasets, methods and unresolved limitations.}\n"
        "\\label{tab:related}\n\\small\n"
        "\\begin{tabular}{p{3.2cm}p{3.2cm}p{3.2cm}p{4cm}}\n\\toprule\n"
        "Study & Dataset & Method & Limitation \\\\\n\\midrule\n"
        + rows + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n"
    )

open(f'{PROJECT}/search_log.tex', 'w').write(search_log_tex(SEARCH_LOG, COUNTS))
open(f'{PROJECT}/extraction.tex', 'w').write(extraction_tex(included))
open(f'{PROJECT}/screening_log.tex', 'w').write(screening_log_tex(SCREENED))
print("wrote search_log.tex, extraction.tex, screening_log.tex\n")
print(search_log_tex(SEARCH_LOG, COUNTS))

wrote search_log.tex, extraction.tex, screening_log.tex

\begin{table}[htbp]
\centering
\caption{Search log. All queries executed against the OpenAlex API.}
\label{tab:searchlog}
\small
\begin{tabular}{p{1.8cm}p{5.2cm}p{2.1cm}rr}
\toprule
Source & Query & Executed & Hits & Retrieved \\
\midrule
OpenAlex & hospital readmission prediction machine learning EHR & 2026-09-21 08:49 & 6429 & 25 \\
OpenAlex & predicting 30 day readmission missing data class imbalance & 2026-09-21 08:49 & 2625 & 25 \\
OpenAlex & clinical dataset machine learning patient demographics readmission & 2026-09-21 08:49 & 8956 & 25 \\
OpenAlex & electronic health records readmission feature engineering machine learning & 2026-09-21 08:49 & 4325 & 25 \\
\bottomrule
\end{tabular}

\vspace{0.5em}
{\small Retrieved $n=100$; after deduplication $n=77$; screened $n=77$; included $n=9$.}
\end{table}



In `paper.tex`, replace the hand-written tables with:

```latex
\input{search_log}
\input{extraction}
```

and put the full screening record in the appendix, where a long table belongs:

```latex
\input{screening_log}
```

Then recompile in the paper notebook. Citations resolve because `refs.bib` and the
`\citet` keys were generated together.

## 9. Optional: Semantic Scholar as a second source

Two sources beat one, and "searched two independent databases" is a real methodological
claim. No key needed, but unauthenticated traffic shares one communal rate limit, so
expect occasional 429s — the retry below handles them. A free personal key removes the
problem if you want to register tonight.

In [11]:
S2 = 'https://api.semanticscholar.org/graph/v1/paper/search'
S2_KEY = ''   # optional; free from semanticscholar.org/product/api

def s2_search(query, n=20, from_year=2005, retries=3):
    headers = {'x-api-key': S2_KEY} if S2_KEY else {}
    params = {'query': query, 'limit': n,
              'fields': 'title,year,abstract,authors,venue,externalIds,citationCount',
              'year': f'{from_year}-'}
    for attempt in range(retries):
        r = requests.get(S2, params=params, headers=headers, timeout=40)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1)); continue
        r.raise_for_status()
        data = r.json().get('data', [])
        rows = []
        for w in data:
            authors = [a.get('name', '') for a in (w.get('authors') or [])][:8]
            last = re.sub(r'\W+', '', authors[0].split()[-1].lower()) if authors else 'anon'
            rows.append({
                'key': f"{last}{w.get('year', 'nd')}", 'title': w.get('title') or '',
                'year': w.get('year'), 'authors': ' and '.join(authors),
                'venue': w.get('venue') or '',
                'doi': (w.get('externalIds') or {}).get('DOI', '') or '',
                'cited_by': w.get('citationCount', 0), 'oa_url': '',
                'abstract': w.get('abstract') or '', 'query': query,
            })
        SEARCH_LOG.append({'query': query, 'source': 'SemanticScholar',
                           'date': date.today().isoformat(), 'filter': f'>= {from_year}',
                           'returned': r.json().get('total', len(rows)), 'retrieved': len(rows)})
        return rows
    print(f"rate limited after {retries} attempts: {query}")
    return []

# extra = []
# for q in QUERIES[:2]:
#     extra += s2_search(q); time.sleep(2)
# papers = dedupe(pd.concat([raw, pd.DataFrame(extra)], ignore_index=True))
# COUNTS['after_dedupe'] = len(papers)